# Event Study — Exploratory Data Analysis

This notebook is for interactive exploration. Run the pipeline scripts first:
```
python src/01_download.py
python src/02_returns.py
python src/03_market_model.py
python src/04_event_study.py
python src/05_export.py
```

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import config

PROC = ROOT / 'data' / 'processed'

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 1. Price levels

In [ ]:
prices = pd.read_csv(PROC / 'prices_aligned.csv', index_col='Date', parse_dates=True)
print(prices.shape)
prices.head()

In [ ]:
# Normalise to 100 at start for comparison
normed = prices[config.TICKERS].div(prices[config.TICKERS].iloc[0]) * 100
ax = normed.plot(title='Normalised price (100 = first obs)', alpha=0.8)
ax.axvline(pd.Timestamp(config.EVENT_DATE), color='red', linestyle='--', label='Event date')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()

## 2. Return distributions

In [ ]:
returns = pd.read_csv(PROC / 'returns.csv', index_col='Date', parse_dates=True)
returns[config.TICKERS].describe().T.round(6)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7), tight_layout=True)
for ax, ticker in zip(axes.flat, config.TICKERS):
    if ticker in returns.columns:
        returns[ticker].hist(bins=60, ax=ax)
        ax.set_title(ticker, fontsize=9)
        ax.set_xlabel('log return')
plt.suptitle('Log-return distributions (full sample)', y=1.02)
plt.show()

## 3. Market model parameters

In [ ]:
params = pd.read_csv(PROC / 'market_model_params.csv', index_col='ticker')
params.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

params['beta'].sort_values().plot.barh(ax=axes[0], color='steelblue')
axes[0].axvline(1, color='red', linestyle='--', alpha=0.5, label='β=1')
axes[0].set_title('Market Beta (β)')
axes[0].legend()

params['r_squared'].sort_values().plot.barh(ax=axes[1], color='darkorange')
axes[1].set_title('R² (estimation window)')

plt.tight_layout()

## 4. Abnormal Returns

In [ ]:
ar = pd.read_csv(PROC / 'abnormal_returns.csv', index_col='t')
ticker_cols = [c for c in ar.columns if c != 'date']
ar[ticker_cols].head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for ticker in ticker_cols:
    ax.plot(ar.index, ar[ticker], marker='o', markersize=3, alpha=0.7, label=ticker)
ax.axhline(0, color='black', linewidth=0.8)
ax.axvline(0, color='red', linestyle='--', linewidth=1.2, label='T=0 (event)')
ax.set_xlabel('Event day (t)')
ax.set_ylabel('Abnormal Return')
ax.set_title(f'Abnormal Returns around {config.EVENT_DATE}')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()

## 5. CAR chart

In [ ]:
car = pd.read_csv(PROC / 'car.csv', index_col='ticker')
car.round(4)

In [ ]:
# Cumulative AR paths
cum_ar = ar[ticker_cols].cumsum()

fig, ax = plt.subplots(figsize=(12, 5))
for ticker in ticker_cols:
    ax.plot(cum_ar.index, cum_ar[ticker], marker='o', markersize=3, alpha=0.7, label=ticker)
ax.axhline(0, color='black', linewidth=0.8)
ax.axvline(0, color='red', linestyle='--', linewidth=1.2, label='T=0 (event)')
ax.set_xlabel('Event day (t)')
ax.set_ylabel('Cumulative Abnormal Return')
ax.set_title(f'CAR paths around {config.EVENT_DATE}')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()

In [ ]:
# Bar chart: final CAR per ticker, colour by significance
colors = [
    'seagreen' if sig else 'steelblue'
    for sig in car['significant_5pct'].values
]
ax = car['CAR'].plot.bar(color=colors, figsize=(10, 4), edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('CAR per ticker (green = significant at 5%)')
ax.set_ylabel('CAR')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()